# Laboratorio 23: Clasificación con Quantum Kernel (ZZFeatureMap)

Comparamos un SVM con kernel cuántico (ZZFeatureMap) frente a un SVM clásico (RBF) en el dataset de lunas entrelazadas.

**Prerequisitos:** Módulo 12 (QML), Lab 27 (QML datos reales), Lab 40 (avanzado).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit.circuit.library import ZZFeatureMap
from qiskit.quantum_info import Statevector
from sklearn.datasets import make_moons
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
print('Entorno listo')

## 1. Dataset: lunas entrelazadas

Problema de clasificación binaria no linealmente separable. 80 muestras, 2 características escaladas a [-π, π].

In [ ]:
# Dataset: lunas entrelazadas (no linealmente separable)
np.random.seed(42)
X, y = make_moons(n_samples=80, noise=0.15)
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

print(f'Train: {len(X_train)} muestras | Test: {len(X_test)} muestras')
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(X_scaled[y==0,0], X_scaled[y==0,1], label='Clase 0', alpha=0.7)
ax.scatter(X_scaled[y==1,0], X_scaled[y==1,1], label='Clase 1', alpha=0.7)
ax.set_title('Dataset: lunas entrelazadas'); ax.legend()
plt.tight_layout(); plt.show()

## 2. El mapa de características cuántico: ZZFeatureMap

ZZFeatureMap codifica x ∈ R² en un estado |φ(x)⟩ de 2 qubits usando puertas Rz y ZZ(x). El kernel es K(x,x') = |⟨φ(x)|φ(x')⟩|².

In [ ]:
# Kernel cuantico: K(x,x') = |<phi(x)|phi(x')>|^2
# El mapa de caracteristicas es ZZFeatureMap de Qiskit

feature_map = ZZFeatureMap(feature_dimension=2, reps=2)
print('ZZFeatureMap circuit:')
print(feature_map.decompose().draw('text'))

## 3. Calcular el kernel cuántico

Calculamos K(xi, xj) para cada par de muestras de entrenamiento. Esto forma la **matriz de Gram** usada por el SVM.

In [ ]:
def quantum_kernel(x1, x2, feature_map):
    'Calcula K(x1,x2) = |<phi(x1)|phi(x2)>|^2 via Statevector.'
    qc1 = feature_map.assign_parameters(dict(zip(feature_map.parameters, x1)))
    qc2 = feature_map.assign_parameters(dict(zip(feature_map.parameters, x2)))
    sv1 = Statevector(qc1)
    sv2 = Statevector(qc2)
    return abs(sv1.inner(sv2))**2

# Construir matrices de Gram (puede tardar ~30s para 60 muestras)
print('Calculando matriz de Gram (train)...')
n_train = len(X_train)
K_train = np.zeros((n_train, n_train))
for i in range(n_train):
    for j in range(i, n_train):
        k = quantum_kernel(X_train[i], X_train[j], feature_map)
        K_train[i,j] = K_train[j,i] = k
print(f'Gram train: {K_train.shape}, diagonal media = {np.diag(K_train).mean():.3f}')

## 4. Entrenar y comparar clasificadores

In [ ]:
print('Calculando matriz de Gram (test)...')
K_test = np.zeros((len(X_test), n_train))
for i in range(len(X_test)):
    for j in range(n_train):
        K_test[i,j] = quantum_kernel(X_test[i], X_train[j], feature_map)

# SVM con kernel cuantico (precomputed)
svm_q = SVC(kernel='precomputed', C=1.0)
svm_q.fit(K_train, y_train)
acc_q = accuracy_score(y_test, svm_q.predict(K_test))

# SVM clasico con kernel RBF
svm_rbf = SVC(kernel='rbf', C=1.0)
svm_rbf.fit(X_train, y_train)
acc_rbf = accuracy_score(y_test, svm_rbf.predict(X_test))

print(f'Accuracy SVM cuantico (ZZFeatureMap): {acc_q:.3f} ({acc_q*100:.1f}%)')
print(f'Accuracy SVM clasico (RBF):           {acc_rbf:.3f} ({acc_rbf*100:.1f}%)')

## 5. Visualización y conclusiones

In [ ]:
# Visualizar la matriz del kernel cuantico
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im1 = axes[0].imshow(K_train, cmap='viridis', aspect='auto')
plt.colorbar(im1, ax=axes[0])
axes[0].set_title('Matriz del Kernel Cuantico K_train')
axes[0].set_xlabel('Muestra j'); axes[0].set_ylabel('Muestra i')

axes[1].bar(['SVM Cuantico\n(ZZFeatureMap)', 'SVM Clasico\n(RBF)'],
            [acc_q, acc_rbf], color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_ylim([0, 1.1])
axes[1].set_ylabel('Accuracy (test)'); axes[1].set_title('Comparativa de Kernels')
for i, v in enumerate([acc_q, acc_rbf]):
    axes[1].text(i, v+0.03, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout(); plt.savefig('kernel_comparison.png', dpi=100)
plt.show()
print('\nConclusión: el kernel cuantico captura correlaciones en el espacio de Hilbert de los circuitos.')